# 03 · Join Sofascore + Capology — Spain La Liga 22/23

Integración de estadísticas de rendimiento (Sofascore) con datos salariales (Capology)
para la temporada **2022/23 de La Liga española**.

**Flujo de matching:**
1. Normalización de nombres (tildes, mayúsculas, caracteres especiales)
2. `TEAM_MAP`: alineación manual de nombres de equipo entre fuentes
3. Merge exacto normalizado
4. Fuzzy matching en cuatro niveles:
   - Score ≥ 0.90 → aceptación automática
   - 0.75 ≤ score < 0.90 → revisión manual
   - 0.50 ≤ score < 0.75 → revisión manual estricta
   - score < 0.50 → revisión manual muy estricta
5. Revisión de jugadores sin salario
6. Guardado en `data/master/`

---

## 1. Imports y rutas

In [1]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import fuzz
from pathlib import Path
from IPython.display import display

ROOT       = Path.cwd().parents[1]
SF_DIR     = ROOT / 'data' / 'processed' / 'sofascore'
CG_DIR     = ROOT / 'data' / 'processed' / 'capology'
MASTER_DIR = ROOT / 'data' / 'master'
MASTER_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Rutas configuradas')
print(f'   Root:   {ROOT}')
print(f'   Master: {MASTER_DIR}')

✅ Rutas configuradas
   Root:   d:\USER\Desktop\TFM
   Master: d:\USER\Desktop\TFM\data\master


## 2. Función de normalización

In [2]:
def normalize(s):
    """
    Normaliza un string para comparación: elimina tildes, pasa a minúsculas,
    elimina caracteres especiales y espacios extra.
    """
    if pd.isna(s):
        return ''
    s = str(s)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    s = re.sub(r'[^a-z0-9\s]', ' ', s.lower().strip())
    return re.sub(r'\s+', ' ', s).strip()

print('✅ Función definida')

✅ Función definida


## 3. Carga de datos

In [3]:
df_sf = pd.read_csv(SF_DIR / 'df_spain_2223.csv').copy()
df_cg = pd.read_csv(CG_DIR / 'cg_spain_2223.csv').copy()

print(f'Sofascore:  {df_sf.shape[0]} jugadores | {df_sf.shape[1]} columnas')
print(f'Capology:   {df_cg.shape[0]} jugadores | {df_cg.shape[1]} columnas')

Sofascore:  584 jugadores | 116 columnas
Capology:   543 jugadores | 9 columnas


## 4. Normalización

In [4]:
df_sf['player_norm'] = df_sf['player'].apply(normalize)
df_sf['team_norm']   = df_sf['team'].apply(normalize)
df_cg['player_norm'] = df_cg['player'].apply(normalize)
df_cg['team_norm']   = df_cg['club'].apply(normalize)

print('✅ Normalización aplicada')

✅ Normalización aplicada


## 5. Alineación de equipos (TEAM_MAP)

### 5.1 Identificar discrepancias de nombres de equipo

In [5]:
solo_sf = set(df_sf['team_norm'].unique()) - set(df_cg['team_norm'].unique())
solo_cg = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())

print('En Sofascore pero no en Capology:')
for e in sorted(solo_sf): print(f'   {e}')
print()
print('En Capology pero no en Sofascore:')
for e in sorted(solo_cg): print(f'   {e}')

En Sofascore pero no en Capology:
   girona fc
   real valladolid

En Capology pero no en Sofascore:
   girona
   valladolid


### 5.2 Aplicar TEAM_MAP

Rellenar con las discrepancias identificadas en la celda anterior.

In [6]:
# ── Ajustar según la celda anterior ──────────────────────────
TEAM_MAP = {'girona':'girona fc',
            'valladolid':'real valladolid'
}
# ─────────────────────────────────────────────────────────────

df_cg['team_norm'] = df_cg['team_norm'].replace(TEAM_MAP)

diff = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())
if diff:
    print(f'⚠️  Equipos de CG aún sin match en SF: {diff}')
else:
    print('✅ Todos los equipos alineados')


✅ Todos los equipos alineados


## 6. Merge exacto normalizado

In [7]:
df_merged = df_sf.merge(
    df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
            'position', 'age', 'nationality']],
    on=['player_norm', 'team_norm'],
    how='left'
)

matched = df_merged['gross_annual_eur'].notna().sum()
total   = len(df_merged)

print(f'Merge exacto: {matched}/{total} ({matched/total:.1%})')
print(f'Sin emparejar: {total - matched}')

Merge exacto: 485/584 (83.0%)
Sin emparejar: 99


## 7. Fuzzy matching sobre los no emparejados

Se generan candidatos para todos los jugadores sin match exacto,
sin umbral mínimo, y se clasifican en cuatro niveles.

In [8]:
df_unmatched = df_merged[df_merged['gross_annual_eur'].isna()].copy()
cg_by_team   = df_cg.groupby('team_norm')['player_norm'].apply(list).to_dict()

rows = []
for _, row in df_unmatched[['player','team','player_norm','team_norm']].drop_duplicates().iterrows():
    candidates = cg_by_team.get(row['team_norm'], [])
    best_match, best_score = None, 0
    for cand in candidates:
        score = fuzz.ratio(row['player_norm'], cand) / 100
        if score > best_score:
            best_score = score
            best_match = cand
    if best_match is not None:
        rows.append({
            'player_sf'  : row['player'],
            'team'       : row['team'],
            'player_norm': row['player_norm'],
            'team_norm'  : row['team_norm'],
            'cg_match'   : best_match,
            'score'      : round(best_score, 3)
        })

df_candidates = pd.DataFrame(rows).sort_values('score', ascending=False)
auto_matches     = df_candidates[df_candidates['score'] >= 0.90].copy()
review_matches   = df_candidates[(df_candidates['score'] >= 0.75) & (df_candidates['score'] < 0.90)].copy()
low_matches      = df_candidates[(df_candidates['score'] >= 0.50) & (df_candidates['score'] < 0.75)].copy()
very_low_matches = df_candidates[df_candidates['score'] < 0.50].copy()

print(f'Auto-aceptados    (score ≥ 0.90):          {len(auto_matches)}')
print(f'Revisión media    (0.75 ≤ score < 0.90):   {len(review_matches)}')
print(f'Revisión estricta (0.50 ≤ score < 0.75):   {len(low_matches)}')
print(f'Revisión muy est. (score < 0.50):           {len(very_low_matches)}')

Auto-aceptados    (score ≥ 0.90):          10
Revisión media    (0.75 ≤ score < 0.90):   11
Revisión estricta (0.50 ≤ score < 0.75):   50
Revisión muy est. (score < 0.50):           28


### 7.1 Matches automáticos (score ≥ 0.90)

Revisar para confirmar que todos son correctos.

In [9]:
auto_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
29,Jørgen Strand Larsen,Celta Vigo,jorgen strand larsen,0.974
22,Alexander Sørloth,Real Sociedad,alexander sorloth,0.970
85,Filip Jørgensen,Villarreal,filip jorgensen,0.966
6,Viktor Tsygankov,Girona FC,viktor tsyhankov,0.938
28,Javier Hernández,Girona FC,javi hernandez,0.933
37,Manuel Sánchez,Osasuna,manu sanchez,0.923
78,Juan Narváez,Real Valladolid,juanjo narvaez,0.923
14,Daniel Vivian,Athletic Club,dani vivian,0.917
10,Yéremy Pino,Villarreal,yeremi pino,0.909
30,Srđan Babić,Almería,srdjan babic,0.909


### 7.2 Revisión media (0.75 ≤ score < 0.90)

Añadir a `EXCLUDE_FROM_FUZZY` el `player_norm` de los incorrectos.

In [10]:
review_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
57,Nicolás Melamed,Espanyol,nico melamed,0.889
60,Josep Chavarría,Rayo Vallecano,pep chavarria,0.857
84,Wassim Keddari,Espanyol,simo keddari,0.846
21,Abdessamad Ezzalzouli,Osasuna,abde ezzalzouli,0.833
58,Nicolás Fernández,Elche,nicolas fernandez mercau,0.829
1,José María Giménez,Atlético Madrid,jose gimenez,0.800
13,Rober González,Real Betis,edgar gonzalez,0.786
45,Mamadou Mbaye,Cádiz,momo mbaye,0.783
2,José Luis Gayà,Valencia,jose gaya,0.783
27,Luis Javier Suárez,Almería,luis suarez,0.759


In [11]:
# ── Falsos positivos a excluir del nivel medio ────────────────
EXCLUDE_FROM_FUZZY = ['rober gonzalez',
                      'alvaro rodriguez'
]
# ─────────────────────────────────────────────────────────────

review_accepted = review_matches[~review_matches['player_norm'].isin(EXCLUDE_FROM_FUZZY)]
print(f'Aceptados: {len(review_accepted)} | Excluidos: {len(EXCLUDE_FROM_FUZZY)}')


Aceptados: 9 | Excluidos: 2


### 7.3 Revisión estricta (0.50 ≤ score < 0.75)

Por defecto ninguno se acepta. Añadir a `ACCEPT_LOW_FUZZY` los correctos.

In [12]:
low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
38,Pathé Ismaël Ciss,Rayo Vallecano,pathe ciss,0.741
3,Jorge Pascual,Villarreal,jorge cuenca,0.720
8,Alejandro Baena,Villarreal,alex baena,0.720
53,Anthony Lozano,Cádiz,antonio blanco,0.714
18,Alexander Isak,Real Sociedad,alexander sorloth,0.710
72,Alberto del Moral,Villarreal,alberto moreno,0.710
96,Diego Mendez,Rayo Vallecano,diego lopez,0.696
32,Hugo Sotelo,Celta Vigo,hugo mallo,0.667
36,Miguel Rodríguez,Celta Vigo,oscar rodriguez,0.645
9,Fernando Medrano,Celta Vigo,fran beltran,0.643


In [13]:
# ── Matches de score bajo confirmados manualmente ─────────────
ACCEPT_LOW_FUZZY = ['pathe ismael ciss',
                    'alejandro baena',
                    'radamel falcao',
                    'pablo gavi',
                    'rodri sanchez',
                    'abner vinicius'                    
]
# ─────────────────────────────────────────────────────────────

low_accepted = low_matches[low_matches['player_norm'].isin(ACCEPT_LOW_FUZZY)]
print(f'Aceptados del nivel bajo: {len(low_accepted)}')


Aceptados del nivel bajo: 6


### 7.4 Revisión muy estricta (score < 0.50)

Por defecto ninguno se acepta. Añadir a `ACCEPT_VERY_LOW_FUZZY` los correctos.

In [14]:
very_low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
16,Jon Ander Olasagasti,Real Sociedad,alexander sorloth,0.486
48,Francisco Mwepu,Cádiz,alfonso espino,0.483
26,Jon Magunazelaia,Real Sociedad,igor zubeldia,0.483
98,Álex Revuelta,Getafe,carles alena,0.480
69,Nabil Touaizi,Espanyol,denis suarez,0.480
46,Oscar Ureña,Girona FC,santiago bueno,0.480
86,Fran Pérez,Valencia,cristian rivero,0.480
95,Pedro Ortiz,Sevilla,kasper dolberg,0.480
71,Javi Llabrés,Mallorca,manu morlanes,0.480
61,Diego Moreno,Osasuna,jon moncayola,0.480


In [15]:
# ── Matches very low confirmados manualmente ──────────────────
ACCEPT_VERY_LOW_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

very_low_accepted = very_low_matches[very_low_matches['player_norm'].isin(ACCEPT_VERY_LOW_FUZZY)]
print(f'Aceptados del nivel very low: {len(very_low_accepted)}')


Aceptados del nivel very low: 0


### 7.5 Aplicar todos los fuzzy matches aceptados

In [16]:
all_fuzzy    = pd.concat([auto_matches, review_accepted, low_accepted, very_low_accepted], ignore_index=True)
fuzzy_lookup = dict(zip(all_fuzzy['player_norm'], all_fuzzy['cg_match']))

df_merged['player_norm_fuzzy'] = df_merged.apply(
    lambda r: fuzzy_lookup.get(r['player_norm'], r['player_norm'])
    if pd.isna(r['gross_annual_eur']) else r['player_norm'],
    axis=1
)

df_final = (
    df_merged
    .drop(columns=['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality'])
    .merge(
        df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
               'position', 'age', 'nationality']],
        left_on=['player_norm_fuzzy', 'team_norm'],
        right_on=['player_norm', 'team_norm'],
        how='left'
    )
    .drop(columns=['player_norm_y', 'player_norm_fuzzy'])
    .rename(columns={'player_norm_x': 'player_norm'})
)

matched_final = df_final['gross_annual_eur'].notna().sum()
print(f'Resultado final: {matched_final}/{len(df_final)} ({matched_final/len(df_final):.1%})')
print(f'Sin salario:     {len(df_final) - matched_final}')

Resultado final: 510/584 (87.3%)
Sin salario:     74


## 8. Revisión de jugadores sin salario

Ordenados por equipo y minutos jugados para identificar si alguno debería tener salario.

In [18]:
sin_salario = (
    df_final[df_final['gross_annual_eur'].isna()]
    [['player', 'team', 'minutesPlayed', 'appearances', 'goals', 'assists']]
    .sort_values(['team', 'minutesPlayed'], ascending=[True, False])
    .reset_index(drop=True)
)

pd.set_option('display.max_rows', None)
print(f'Total sin salario: {len(sin_salario)}')
display(sin_salario)
pd.reset_option('display.max_rows')

Total sin salario: 74


,player,team,minutesPlayed,appearances,goals,assists
0,Curro Sánchez,Almería,41,2,0,0
1,Malcom Ares,Athletic Club,93,8,0,0
2,Carlos Martín,Atlético Madrid,29,4,0,0
3,Antonio Gomis,Atlético Madrid,11,1,0,0
4,Ángel Alarcón,Barcelona,28,4,0,0
5,Lamine Yamal,Barcelona,11,1,0,0
6,Aleix Garrido,Barcelona,9,1,0,0
7,Pierre-Emerick Aubameyang,Barcelona,8,1,0,0
8,Chadi Riad,Barcelona,1,1,0,0
9,Miguel Rodríguez,Celta Vigo,250,7,1,1


### 8.1 Comparación manual por equipo

Para cada equipo con jugadores sin salario se muestra la plantilla completa de Capology
ordenada alfabéticamente por nombre normalizado, facilitando la detección visual de matches fallidos.

In [19]:
equipos_sin_salario = sin_salario['team'].unique()

for equipo in sorted(equipos_sin_salario):
    sf_jugadores = sin_salario[sin_salario['team'] == equipo][['player', 'minutesPlayed']].sort_values('player')

    equipo_norm  = normalize(equipo)
    cg_jugadores = (
        df_cg[df_cg['team_norm'] == equipo_norm][['player', 'player_norm']]
        .sort_values('player_norm')
        .reset_index(drop=True)
    )

    print(f'\n{"="*60}')
    print(f'  {equipo}  —  SF sin salario:')
    display(sf_jugadores.reset_index(drop=True))
    print(f'  CG plantilla completa:')
    display(cg_jugadores)


  Almería  —  SF sin salario:


,player,minutesPlayed
0,Curro Sánchez,41


  CG plantilla completa:


,player,player_norm
0,Adrián Embarba,adrian embarba
1,Alejandro Pozo,alejandro pozo
2,Álex Centelles,alex centelles
3,Arnau Puigmal,arnau puigmal
4,César de la Hoz,cesar de la hoz
5,Chumi,chumi
6,Diego Fuoli,diego fuoli
7,Diego Mariño,diego marino
8,Dyego Sousa,dyego sousa
9,El Bilal Touré,el bilal toure



  Athletic Club  —  SF sin salario:


...

,player,player_norm
0,André Almeida,andre almeida
1,Cenk Özkacar,cenk ozkacar
2,Cristhian Mosquera,cristhian mosquera
3,Cristian Rivero,cristian rivero
4,Dimitri Foulquier,dimitri foulquier
5,Edinson Cavani,edinson cavani
6,Eray Cömert,eray comert
7,Gabriel Paulista,gabriel paulista
8,Giorgi Mamardashvili,giorgi mamardashvili
9,Hugo Duro,hugo duro



  Villarreal  —  SF sin salario:


,player,minutesPlayed
0,Alberto del Moral,9
1,Diego Collado,17
2,Fernando Niño,24
3,Haissem Hassan,50
4,Jorge Pascual,14
5,Mamadou Mbacke,29


  CG plantilla completa:


,player,player_norm
0,Aïssa Mandi,aissa mandi
1,Alberto Moreno,alberto moreno
2,Álex Baena,alex baena
3,Alfonso Pedraza,alfonso pedraza
4,Arnaut Danjuma,arnaut danjuma
5,Dani Parejo,dani parejo
6,Étienne Capoue,etienne capoue
7,Filip Jörgensen,filip jorgensen
8,Francis Coquelin,francis coquelin
9,Gerard Moreno,gerard moreno


In [26]:
# ── Matches manuales (nombres muy distintos o traspasos invernales) ──
# Formato: (player_norm_sf, team_norm_sf): (player_norm_cg, team_norm_cg)
MANUAL_MATCHES = {
    ('anthony lozano', 'cadiz'): ('choco lozano', 'cadiz'),
}
# ────────────────────────────────────────────────────────────────────
print(f'Matches manuales definidos: {len(MANUAL_MATCHES)}')


Matches manuales definidos: 1


In [27]:
# Aplicar matches manuales sobre los que siguen sin salario
for (p_sf, t_sf), (p_cg, t_cg) in MANUAL_MATCHES.items():
    mask = (df_final['player_norm'] == p_sf) & (df_final['team_norm'] == t_sf) & (df_final['gross_annual_eur'].isna())
    datos_cg = df_cg[(df_cg['player_norm'] == p_cg) & (df_cg['team_norm'] == t_cg)]
    if not datos_cg.empty and mask.any():
        for col in ['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality']:
            df_final.loc[mask, col] = datos_cg[col].values[0]
        print(f'✅ Match manual aplicado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')
    else:
        print(f'⚠️  No encontrado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')

matched_tras_manual = df_final['gross_annual_eur'].notna().sum()
print(f'\nTras matches manuales: {matched_tras_manual}/{len(df_final)} ({matched_tras_manual/len(df_final):.1%})')

✅ Match manual aplicado: anthony lozano (cadiz) → choco lozano (cadiz)

Tras matches manuales: 511/584 (87.5%)


## 9. Guardado

Una vez revisado todo, se eliminan las columnas auxiliares y se guarda en `data/master/`.

In [28]:
df_final = df_final.drop(columns=['player_norm', 'team_norm'])

nombre_salida = 'master_spain_2223.csv'
df_final.to_csv(MASTER_DIR / nombre_salida, index=False)

print(f'✅ Guardado: {nombre_salida}')
print(f'   Jugadores totales:  {len(df_final)}')
print(f'   Con salario:        {df_final["gross_annual_eur"].notna().sum()}')
print(f'   Sin salario (NaN):  {df_final["gross_annual_eur"].isna().sum()}')
print(f'   Columnas:           {df_final.shape[1]}')

✅ Guardado: master_spain_2223.csv
   Jugadores totales:  584
   Con salario:        511
   Sin salario (NaN):  73
   Columnas:           121
